In [ ]:
import os, sys, requests
from pyspark.sql import SparkSession;
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


Métodos para download de dados do IBGE e transformação em uma Spark Dataframe

In [ ]:
def get_data_IBGE_Sidra(url):
    try:
        # 1. Faz a requisição HTTP para a API do IBGE
        response = requests.get(url)
        response.raise_for_status()  # Garante que a requisição funcionou (Status 200)

        # 2. Converte a resposta bruta para o formato JSON
        dados_json = response.json()

        return dados_json

    except Exception as e:
        print(e)

def convert_data_to_DF(dados_json):
    # Extração de dados que contém cabeçalho
    # 1. Separa o cabeçalho do restante dos dados
    cabecalho    = dados_json[0]
    linhas_dados = dados_json[1:]

    # 2. Mapeia quais chaves nós queremos extrair baseado no "2º atributo" (as chaves terminadas em 'N' + a chave 'V' de Valor)
    # Isso vai gerar um de-para como: {'D1N': 'Brasil', 'D2N': 'Variável', ..., 'V': 'Valor'}
    # mapeamento_colunas = {k: v for k, v in cabecalho.items() if k.endswith('N') or k == 'V'}
    mapeamento_colunas = {k: v for k, v in cabecalho.items()}

    # 3. Transforma as linhas de dados mantendo apenas as colunas desejadas e já renomeando as chaves
    dados_processados = []
    for linha in linhas_dados:
        nova_linha = {mapeamento_colunas[chave]: valor for chave, valor in linha.items() if chave in mapeamento_colunas}
        dados_processados.append(nova_linha)

    # 4. Cria o DataFrame do Spark diretamente a partir da lista de dicionários limpa
    df = spark.createDataFrame(dados_processados)

    return df

Seleciona somente os níveis de ensino, pois na próxima etapa os dados e cada nivel serão requisitados 

In [ ]:
url = "https://apisidra.ibge.gov.br/values/t/10059/n1/all/v/1013284/p/all/c11798/all/c58/95253/c2/6794/c86/95251/d/v1013284%202"

dados_json = get_data_IBGE_Sidra(url)
df_nivel_ensino = convert_data_to_DF(dados_json)

# remonear colunas para padrão snake_case 
df_nivel_ensino = \
    (df_nivel_ensino
        .withColumnsRenamed({"Nível de ensino ou curso que frequentavam (Código)": "codigo_nivel_ensino"
                            ,"Nível de ensino ou curso que frequentavam": "descricao_nivel_ensino" }))

df_nivel_ensino = \
    (df_nivel_ensino
        .select("codigo_nivel_ensino"
               ,"descricao_nivel_ensino")
    )


In [ ]:

df_nivel_ensino.show(truncate=False)

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 60959)
Traceback (most recent call last):
  File "C:\Users\DRT90628\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "C:\Users\DRT90628\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "C:\Users\DRT90628\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Users\DRT90628\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 755, in __init__
    self.handle()
  File "c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\accumulators.py", line 329, in handle
    poll(accum_updates)
  File "c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark

Os dados serão requisitados para cada nível de ensino pois selecionar todos os níveis e municípios estoura a capacidade determinada pelo IBGE (até 50.000 registros) 

In [ ]:
url = "https://apisidra.ibge.gov.br/values/t/10059/n6/all/v/1013284/p/all/c11798/{codigo_nivel_ensino}/c58/95253/c2/6794/c86/95251/d/v1013284%202"

ls_nivel_ensino = df_nivel_ensino.toLocalIterator()

for rec, nivel_ensino in enumerate(ls_nivel_ensino):

    url_codigo_nivel_ensino = url.format(codigo_nivel_ensino = str(nivel_ensino["codigo_nivel_ensino"]))

    print(url_codigo_nivel_ensino)

    dados_json = get_data_IBGE_Sidra(url_codigo_nivel_ensino)
    df_educacao_nivel_ensino = convert_data_to_DF(dados_json)

    if rec == 0:
        df_educacao = df_educacao_nivel_ensino
    else:
        df_educacao = df_educacao.union(df_educacao_nivel_ensino)



In [ ]:
df_educacao.printSchema()
df_educacao.limit(10).show(truncate=False)


In [ ]:
# Dados sobre educação
# Tabela 10058 - Pessoas de 6 a 17 anos de idade que frequentavam escola, por nível de ensino, segundo os grupos de idade, o sexo e a cor ou raça
url = "https://apisidra.ibge.gov.br/values/t/10058/n6/all/v/1013283/p/all/c11798/allxt/c58/95253/c2/6794/c86/95251/d/v1013283%202"

# Tabela 10059 - Pessoas de 18 anos ou mais de idade que frequentavam escola, por nível de ensino, segundo os grupos de idade, o sexo e a cor ou raça
url = "https://apisidra.ibge.gov.br/values/t/10059/n3/35/v/1013284/p/all/c11798/allxt/c58/95253/c2/6794/c86/95251/d/v1013284%202"
url = "https://apisidra.ibge.gov.br/values/t/10059/n1/all/n6/all/v/1013284/p/all/c11798/7905,7906/c58/95253/c2/6794/c86/95251/d/v1013284%202"

url = "https://apisidra.ibge.gov.br/values/t/10059/n1/all/v/1013284/p/all/c11798/all/c58/95253/c2/6794/c86/95251/d/v1013284%202"

try:
    # 1. Faz a requisição HTTP para a API do IBGE
    response = requests.get(url)
    response.raise_for_status()  # Garante que a requisição funcionou (Status 200)

    # 2. Converte a resposta bruta para o formato JSON
    dados_json = response.json()

    print(len(dados_json))
except Exception as e:
    print(e)

In [ ]:
dados_json

In [ ]:
# Extração de dados que contém cabeçalho

# 1. Separa o cabeçalho do restante dos dados
cabecalho    = dados_json[0]
linhas_dados = dados_json[1:]

# 2. Mapeia quais chaves nós queremos extrair baseado no "2º atributo" (as chaves terminadas em 'N' + a chave 'V' de Valor)
# Isso vai gerar um de-para como: {'D1N': 'Brasil', 'D2N': 'Variável', ..., 'V': 'Valor'}
# mapeamento_colunas = {k: v for k, v in cabecalho.items() if k.endswith('N') or k == 'V'}
mapeamento_colunas = {k: v for k, v in cabecalho.items()}


# 3. Transforma as linhas de dados mantendo apenas as colunas desejadas e já renomeando as chaves
dados_processados = []
for linha in linhas_dados:
    nova_linha = {mapeamento_colunas[chave]: valor for chave, valor in linha.items() if chave in mapeamento_colunas}
    dados_processados.append(nova_linha)

# 4. Cria o DataFrame do Spark diretamente a partir da lista de dicionários limpa
df = spark.createDataFrame(dados_processados)



In [ ]:
df = df.withColumnRenamed("Município (Código)", "codigo_municipio")

In [ ]:
# Exibe o resultado final estruturado
# df.filter("codigo_municipio rlike '110001'").limit(100).show(truncate=False)
df.select("Nível de ensino ou curso que frequentavam (Código)", "Nível de ensino ou curso que frequentavam").show(10, truncate=False)